In [53]:
import csv

def cargar_datos():
    grafo = {}
    heuristicas = {}

    # Leer Heurísticas
    with open('heuristica.csv', mode='r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            heuristicas[row['Activity'].strip()] = int(row['Recovery time after burning 300cal (minutes)'])

    # Leer Costos (Grafo)
    with open('funcion_de_costo.csv', mode='r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            orig = row['Origen'].strip()
            dest = row['Destino'].strip()
            cost = int(row['Cost'])
            if orig not in grafo: grafo[orig] = []
            grafo[orig].append({'dest': dest, 'cost': cost})
            
    return grafo, heuristicas

grafo, heuristicas = cargar_datos()
print("Datos cargados correctamente.")

Datos cargados correctamente.


In [ ]:
class TreeNode:
   def __init__(self, state, path_cost=0, heuristic=0, parent=None):
      self.state = state          
      self.path_cost = path_cost  
      self.heuristic = heuristic  
      self.parent = parent        
      self.children = []        

   def add_child(self, child_node):
      #Metodo que conecta a los nodos en el arbol
      self.children.append(child_node)  

   def costo_total(self):
      #Calculamos f(n) = g(n) + h(n)
      return self.path_cost + self.heuristic

   def __repr__(self):
      # Representación en texto para poder imprimirlo en consola
      return f"<{self.state}: g={self.path_cost}, h={self.heuristic}, f={self.costo_total()}>"

In [55]:
#fifo
class FIFOQueue:
   def __init__(self):
      self.items = []
   
   def EMPTY(self):
      if len(self.items) == 0:
         return True
      return False
   
   def TOP(self):
      if not self.EMPTY():
         return self.items[0]
      return None
   
   def POP(self):
      if not self.EMPTY():
         return self.items.pop(0)
      return None
   
   def ADD(self, item):
      self.items.append(item)
      return self.items

In [56]:
#lifo
class LIFOQueue:
   def __init__(self):
      self.items = []
   
   def EMPTY(self):
      if len(self.items) == 0:
         return True
      return False
   
   def TOP(self):
      if not self.EMPTY():
         return self.items[-1]
      return None
   
   def POP(self):
      if not self.EMPTY():
         return self.items.pop()
      return None
   
   def ADD(self, item):
      self.items.append(item)
      return self.items

In [57]:
class PriorityQueue:
   def __init__(self, use_heuristic=False):
      self.items = []
      self.use_heuristic = use_heuristic 

   def EMPTY(self):
      if len(self.items) == 0:
         return True
      return False
   
   def TOP(self):
      if not self.EMPTY():
         return self.items[0]
      return None
   
   def POP(self):
      if not self.EMPTY():
         return self.items.pop(0)
      return None
   
   def ADD(self, item):
      self.items.append(item)
      if self.use_heuristic:
         self.items.sort(key=lambda node: node.f())
      else:
         self.items.sort(key=lambda node: node.path_cost)
      return self.items

In [58]:
class GreedyQueue(PriorityQueue):
   def ADD(self, item):
      self.items.append(item)
      self.items.sort(key=lambda node: node.heuristic)
      return self.items

In [59]:
#prueba, no tomar e cuenta
def busqueda_gym(inicio, meta, grafo, heuristicas, usar_heuristica=False):
   nodo_raiz = TreeNode(inicio, 0, heuristicas.get(inicio, 0))
   frontera = PriorityQueue(use_heuristic=usar_heuristica)
   frontera.ADD(nodo_raiz)
   
   visitados = set()
   iteracion = 0

   print(f"--- Iniciando Búsqueda ({'A*' if usar_heuristica else 'UCS'}) ---")

   while not frontera.EMPTY():
      nodo_actual = frontera.POP()
      print(f"Iteración {iteracion}: Expandiendo nodo {nodo_actual}")
      iteracion += 1
      
      # Si llegamos a la meta
      if nodo_actual.state == meta:
         print(f"¡Meta encontrada en iteración {iteracion}!")
         return reconstruir_ruta(nodo_actual), nodo_actual.path_cost

      visitados.add(nodo_actual.state)

      # Expandir hijos
      for conexion in grafo.get(nodo_actual.state, []):
         nombre_hijo = conexion['dest']
         costo_arco = conexion['cost']
         
         if nombre_hijo not in visitados:
               hijo = TreeNode(
                  state=nombre_hijo,
                  path_cost=nodo_actual.path_cost + costo_arco,
                  heuristic=heuristicas.get(nombre_hijo, 0),
                  parent=nodo_actual
               )
               nodo_actual.add_child(hijo)
               frontera.ADD(hijo)
               
   return None, 0

def reconstruir_ruta(nodo):
   ruta = []
   actual = nodo
   while actual:
      ruta.append(actual.state)
      actual = actual.parent
   return ruta[::-1] # Invertir para que vaya de inicio a fin

In [60]:
#pruebas
inicio = "Warm-up activities"
meta = "Stretching"

# Ejecutar A*
ruta, costo = busqueda_gym(inicio, meta, grafo, heuristicas, usar_heuristica=True)

print(f"\nRuta recomendada: {' a '.join(ruta)}")
print(f"Costo total de la rutina: {costo} minutos")

--- Iniciando Búsqueda (A*) ---
Iteración 0: Expandiendo nodo <Warm-up activities: g=0, h=5, f=5>
Iteración 1: Expandiendo nodo <Exercise bike: g=10, h=10, f=20>
Iteración 2: Expandiendo nodo <Tread Mill: g=10, h=12, f=22>
Iteración 3: Expandiendo nodo <Step Mill: g=10, h=14, f=24>
Iteración 4: Expandiendo nodo <Skipping Rope: g=10, h=16, f=26>
Iteración 5: Expandiendo nodo <Incline Bench: g=26, h=8, f=34>
Iteración 6: Expandiendo nodo <Dumbbell: g=25, h=9, f=34>
Iteración 7: Expandiendo nodo <Barbell: g=25, h=10, f=35>
Iteración 8: Expandiendo nodo <Incline Bench: g=30, h=8, f=38>
Iteración 9: Expandiendo nodo <Pulling Bars: g=30, h=10, f=40>
Iteración 10: Expandiendo nodo <Climbing Rope: g=36, h=5, f=41>
Iteración 11: Expandiendo nodo <Cable-Crossover: g=35, h=8, f=43>
Iteración 12: Expandiendo nodo <Leg Press Machine: g=35, h=8, f=43>
Iteración 13: Expandiendo nodo <Leg Press Machine: g=37, h=8, f=45>
Iteración 14: Expandiendo nodo <Stretching: g=46, h=0, f=46>
¡Meta encontrada en i

In [61]:
def algoritmo_busqueda(inicio, meta, grafo, heuristicas, tipo_cola, usar_h=False):
   # Creamos el nodo raíz
   h_inicial = heuristicas.get(inicio, 0)
   nodo_raiz = TreeNode(inicio, path_cost=0, heuristic=h_inicial)
   
   # Inicializamos la frontera según el algoritmo
   frontera = tipo_cola
   frontera.ADD(nodo_raiz)
   
   visitados = set()
   nodos_expandidos = 0

   while not frontera.EMPTY():
      nodo_actual = frontera.POP()
      nodos_expandidos += 1
      
      # Objetivo alcanzado
      if nodo_actual.state == meta:
         return reconstruir_ruta(nodo_actual), nodo_actual.path_cost, nodos_expandidos

      if nodo_actual.state not in visitados:
         visitados.add(nodo_actual.state)
         
         # Expandir hijos
         for conexion in grafo.get(nodo_actual.state, []):
               nombre_hijo = conexion['dest']
               costo_arco = conexion['cost']
               
               h_hijo = heuristicas.get(nombre_hijo, 0)
               
               hijo = TreeNode(
                  state=nombre_hijo,
                  path_cost=nodo_actual.path_cost + costo_arco,
                  heuristic=h_hijo,
                  parent=nodo_actual
               )
               nodo_actual.add_child(hijo)
               frontera.ADD(hijo)
               
   return None, 0, nodos_expandidos

def reconstruir_ruta(nodo):
   path = []
   while nodo:
      path.append(nodo.state)
      nodo = nodo.parent
   return " a ".join(path[::-1])

In [62]:
#Implementacion de los 5 algoritmos 
inicio = "Warm-up activities"
meta = "Stretching"

# BFS (Breadth-First Search)
ruta_bfs, costo_bfs, exp_bfs = algoritmo_busqueda(inicio, meta, grafo, heuristicas, FIFOQueue())

# DFS (Depth-First Search)
ruta_dfs, costo_dfs, exp_dfs = algoritmo_busqueda(inicio, meta, grafo, heuristicas, LIFOQueue())

# UCS (Uniform Cost Search)
# PriorityQueue con use_heuristic=False ordena solo por g(n)
ruta_ucs, costo_ucs, exp_ucs = algoritmo_busqueda(inicio, meta, grafo, heuristicas, PriorityQueue(use_heuristic=False))

# Greedy Search
ruta_greedy, costo_greedy, exp_greedy = algoritmo_busqueda(inicio, meta, grafo, heuristicas, GreedyQueue())

# A* Search
# PriorityQueue con use_heuristic=True ordena por f(n) = g(n) + h(n)
ruta_astar, costo_astar, exp_astar = algoritmo_busqueda(inicio, meta, grafo, heuristicas, PriorityQueue(use_heuristic=True))

In [63]:
print(f"{'ALGORITMO':<15} | {'COSTO':<7} | {'NODOS EXP':<10} | {'RUTA'}")
print("-" * 80)
print(f"{'BFS':<15} | {costo_bfs:<7} | {exp_bfs:<10} | {ruta_bfs}")
print(f"{'DFS':<15} | {costo_dfs:<7} | {exp_dfs:<10} | {ruta_dfs}")
print(f"{'UCS':<15} | {costo_ucs:<7} | {exp_ucs:<10} | {ruta_ucs}")
print(f"{'Greedy':<15} | {costo_greedy:<7} | {exp_greedy:<10} | {ruta_greedy}")
print(f"{'A*':<15} | {costo_astar:<7} | {exp_astar:<10} | {ruta_astar}")

ALGORITMO       | COSTO   | NODOS EXP  | RUTA
--------------------------------------------------------------------------------
BFS             | 48      | 17         | Warm-up activities a Skipping Rope a Dumbbell a Leg Press Machine a Stretching
DFS             | 54      | 5          | Warm-up activities a Step Mill a Incline Bench a Hammer Strength a Stretching
UCS             | 46      | 17         | Warm-up activities a Skipping Rope a Barbell a Leg Press Machine a Stretching
Greedy          | 55      | 5          | Warm-up activities a Exercise bike a Cable-Crossover a Climbing Rope a Stretching
A*              | 46      | 15         | Warm-up activities a Tread Mill a Pulling Bars a Climbing Rope a Stretching
